# Tideway Events & Knowledge API Examples

Helper methods paired with direct `get`/`post` calls for events and knowledge endpoints. Reference: https://docs.bmc.com/xwiki/bin/view/IT-Operations-Management/Discovery/BMC-Helix-Discovery/DAAS/Integrating/Using-the-REST-APIs/Endpoints-in-the-REST-API/.


## Setup
- Install tideway (e.g. `pip install -e .` from repo root).
- Set `TARGET` and `TOKEN` below. Do **not** commit secrets.
- Adjust request bodies and file paths to match your appliance.


In [ ]:
import tideway
from pathlib import Path

# Configuration
TARGET = 'appliance-hostname-or-ip'  # e.g. 'discovery.example.com'
TOKEN = 'your-api-token'            # keep secrets out of source control
API_VERSION = '1.16'                # latest supported API
SSL_VERIFY = False                  # set to True when using valid certs

tw = tideway.appliance(TARGET, TOKEN, api_version=API_VERSION, ssl_verify=SSL_VERIFY)
events = tw.events()
knowledge = tw.knowledge()

def show_response(resp, label):
    if resp.ok:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    return {'endpoint': label, 'status': resp.status_code, 'body': body}

tw.api_about.json()  # quick sanity check


## Events
Send events using helper vs direct POST.


In [ ]:
event_body = {
    # 'status': 'warning',
    # 'message': 'Example event from notebook',
    # 'source': 'notebook',
}

if event_body:
    event_helper = events.post_events(event_body)
    event_direct = events.post('/events', event_body)
    event_calls = {
        '/events (helper POST)': show_response(event_helper, '/events'),
        '/events (direct POST)': show_response(event_direct, '/events'),
    }
else:
    event_calls = 'Set event_body to post /events.'
event_calls


## Knowledge state
Helpers vs direct GET for knowledge info and upload status.


In [ ]:
knowledge_helper = knowledge.get_knowledge_property
knowledge_direct = knowledge.get('/knowledge')

knowledge_status_helper = knowledge.get_knowledge_status_property
knowledge_status_direct = knowledge.get('/knowledge/status')

knowledge_calls = {
    '/knowledge (helper)': show_response(knowledge_helper, '/knowledge'),
    '/knowledge (direct GET)': show_response(knowledge_direct, '/knowledge'),
    '/knowledge/status (helper)': show_response(knowledge_status_helper, '/knowledge/status'),
    '/knowledge/status (direct GET)': show_response(knowledge_status_direct, '/knowledge/status'),
}
knowledge_calls


## Upload knowledge (TKU/pattern module)
Provide a filename and local path to upload; set `activate`/`allow_restart` flags per your process.


In [ ]:
upload_filename = 'tku.zip'   # name shown in the endpoint URL
upload_path = ''              # local path to the TKU file
activate = True
allow_restart = False

if upload_path and Path(upload_path).exists():
    upload_helper = knowledge.post_knowledge(upload_filename, upload_path, activate=activate, allow_restart=allow_restart)
    # Direct call mirrors the helper: set params then send multipart file
    knowledge.params['activate'] = activate
    knowledge.params['allow_restart'] = allow_restart
    with open(upload_path, 'rb') as upload_file:
        upload_direct = knowledge.post(f"/knowledge/{upload_filename}", files={'file': upload_file}, response='text/html')
    upload_calls = {
        f"/knowledge/{upload_filename} (helper POST)": show_response(upload_helper, f"/knowledge/{upload_filename}"),
        f"/knowledge/{upload_filename} (direct POST)": show_response(upload_direct, f"/knowledge/{upload_filename}"),
    }
else:
    upload_calls = 'Set upload_path to a TKU file to post /knowledge/{filename}.'
upload_calls


## Trigger patterns
List knowledge trigger patterns via helper vs direct GET.


In [ ]:
lookup_data_sources = None  # e.g. True to include lookup data sources

# Helper allows passing lookup_data_sources param
trigger_patterns_helper = knowledge.get_knowledge_trigger_patterns(lookup_data_sources=lookup_data_sources)

# Direct GET (set params manually if you need lookup_data_sources)
if lookup_data_sources is not None:
    knowledge.params['lookup_data_sources'] = lookup_data_sources
trigger_patterns_direct = knowledge.get('/knowledge/trigger_patterns')

trigger_calls = {
    '/knowledge/trigger_patterns (helper)': show_response(trigger_patterns_helper, '/knowledge/trigger_patterns'),
    '/knowledge/trigger_patterns (direct)': show_response(trigger_patterns_direct, '/knowledge/trigger_patterns'),
}
trigger_calls
